# Knowledge Catalog programmatic search, deep inspection, and AI agent grounding

<a href="https://colab.research.google.com/github/GoogleCloudPlatform/knowledge-catalog/blob/main/cookbooks/catalog_search_and_retrieval.ipynb?utm_source=devrel&utm_medium=colab_badge&utm_campaign=catalog_search_and_retrieval" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Section 1: Executive overview, scenario & architectural blueprint

Modern enterprise data platforms face two persistent metadata governance hurdles:
1. **Unmasked PII exposure across analytical datasets**: Compliance stewards cannot manually inspect thousands of table schemas to confirm sensitive columns are de-identified. While catalog search locates candidate tables, search index entries return empty aspect maps by design, leaving stewards unable to audit column-level protection policies without chaining discovery to deep entry lookups.
2. **Relational query hallucinations in autonomous AI agents**: Autonomous analytical agents querying enterprise data warehouses hallucinate non-existent column names and synthesize invalid cross-table join keys when supplied with raw prompt instructions. Without authoritative catalog context (column types, semantic descriptions, and historical query join paths), agents generate broken SQL and erode stakeholder trust.

This cookbook implements an authentic metadata discovery, compliance auditing, and agent grounding loop using **Knowledge Catalog**, **BigQuery**, and **Gemini 3.7 Flash** (via the unified Google GenAI SDK).

```
┌────────────────────────────────────────────────────────────────────────────────────────────────────────┐
│                        KNOWLEDGE CATALOG 3-TIER SEARCH & RETRIEVAL ARCHITECTURE                        │
├────────────────────────────────────────────────────────────────────────────────────────────────────────┤
│                                                                                                        │
│  [1. PHYSICAL ASSETS]      BigQuery thelook_ecommerce in US multi-region (DATA_LOCATION="us")          │
│                            • users (unmasked PII: email, first_name, last_name, street_address)        │
│                            • orders, order_items, products                                             │
│                                                   │                                                    │
│                                                   ▼                                                    │
│  [2. GOVERNANCE BINDING]   Knowledge Catalog custom global AspectType 'pii' (KC_LOCATION="us")         │
│                            • 7 column-level aspect bindings (4 unmasked violations, 3 masked)          │
│                            • Bounded headless polling loop verifying search index propagation          │
│                                                   │                                                    │
│         ┌─────────────────────────────────────────┴─────────────────────────────────────────┐          │
│         ▼                                                                                   ▼          │
│  [3. TIER 1: DISCOVERY]                                              [4. TIER 2: INSPECTION]           │
│  search_entries (locations/global)                                   lookup_entry (locations/us)       │
│  • Semantic search: 2-stage neural ANN (~100 candidate cap)          • view = EntryView.CUSTOM         │
│  • Predicate search: Spanner scan (unbounded pagination)             • aspect_types = [pii_aspect]     │
│  • Proves search aspects: {} map is intentionally empty              • Client-side post-filter detects │
│                                                                        4 unmasked PII violations       │
│                                                                                             │          │
│                                                                                             ▼          │
│  [5. TIER 3: HYDRATION & GROUNDING]                                                                    │
│  lookup_context & Gemini Enterprise Agent Platform                                                    │
│  • Defensive candidate guard intercepts empty lists (prevents CONTEXT_RESOURCES_EMPTY backend crash)   │
│  • Hydrates token-efficient YAML schemas and certified join telemetry (order_items ── products)       │
│  • Gemini 3.7 Flash (GEMINI_LOCATION="global") synthesizes certified SQL join with zero hallucinations│
└────────────────────────────────────────────────────────────────────────────────────────────────────────┘
```

### Target persona

* **Enterprise data platform engineers and governance stewards**: Designing automated metadata discovery, compliance auditing, and sensitive data policy enforcement across the lakehouse.
* **Applied AI and data analytics engineers**: Equipping autonomous analytical agents with authoritative catalog schemas and join telemetry to eliminate relational SQL hallucinations.

### Prerequisites

* A Google Cloud project with billing enabled.
* Required IAM permissions: Knowledge Catalog Admin (`roles/dataplex.catalogAdmin`), BigQuery Admin (`roles/bigquery.admin`), and Gemini model invocation permissions.
* Python 3.10+ in a Jupyter or Google Cloud Colab environment.

### Estimated time

* **25 minutes**

### Measurable learning objectives

1. Execute semantic and predicate-driven catalog searches using Knowledge Catalog global control-plane endpoints, evaluating the trade-offs between two-stage neural candidate generation and unbounded predicate Spanner index scans.
2. Audit column-level governance metadata by chaining search results to deep entry lookups with custom entry views and client-side post-filtering to detect unmasked PII columns, resolving the empty search aspects invariant.
3. Hydrate prompt-ready operational schemas using Knowledge Catalog context retrieval bounded by character budget with defensive empty-candidate guards, extracting multi-table schemas and frequent join paths in YAML format.
4. Ground an autonomous analytical AI agent using Gemini 3.7 Flash and the unified Google GenAI SDK to resolve natural-language data queries against verified catalog join paths with zero hallucinations.
5. Intercept search index eventual consistency latency using programmatic bounded polling loops to guarantee deterministic execution across automated pipelines.

In [ ]:
# Authenticate user session when running in Google Colab
import sys

if "google.colab" in sys.modules:
    from google.colab import auth

    auth.authenticate_user()
    print("Google Colab user authentication succeeded.")
else:
    print("Running outside Google Colab; using Application Default Credentials (ADC).")


---

## Section 2: Environment setup & parameterized guardrails

Prepare your execution environment by installing the required Google Cloud client libraries, the unified Google GenAI SDK, and Pydantic.

### Native client exposure vs. proprietary wrappers

To build production-grade enterprise data pipelines, you interact directly with official Google Cloud client libraries (`google-cloud-dataplex`, `google-cloud-bigquery`, `google-genai`). Exposing native SDKs directly gives you full visibility into authentication flows, method parameters, and low-level response payloads without the cognitive opacity of bespoke framework wrappers.

When running in Google Colab, the notebook authenticates your interactive user session via `google.colab.auth.authenticate_user()`. When executing in a local terminal, Cloud Shell, or automated CI/CD pipeline, client libraries automatically discover credentials via Application Default Credentials (ADC).

> ℹ️ **Authentication Portability**: Inspecting `sys.modules` allows the notebook to run seamlessly across interactive Colab sessions and headless enterprise development environments without requiring code modifications or hardcoded service account keys.

In [ ]:
# Install required Google Cloud SDKs, Google GenAI SDK, and Pydantic
%pip install -q google-cloud-dataplex google-cloud-bigquery google-genai pydantic


### Parameter configuration and data residency

Configure your Google Cloud project ID and deployment parameters.

To enforce deterministic parameter contracts and prevent partial-execution failures, parameters are defined once as canonical constants:
* `PROJECT_ID`: The single configuration variable decorated with `@param`. A fail-fast guard raises an immediate `ValueError` if the placeholder string remains unmodified.
* `DATA_LOCATION`: BigQuery dataset storage location set to `"us"` multi-region to align natively with public data source `bigquery-public-data.thelook_ecommerce`, eliminating cross-datacenter egress latency and data transfer costs.
* `KC_LOCATION`: Regional data plane for Knowledge Catalog entries, matching BigQuery storage (`"us"`).
* `GEMINI_LOCATION`: Decoupled inference endpoint set to `"global"` for the flagship Gemini model.
* `MODEL_NAME`: Active flagship model (`"gemini-3.7-flash"`) for structured reasoning and tool calling.
* `DATASET_ID`: Analytical dataset name (`"thelook_ecommerce"`) provisioned with a 24-hour auto-expiration lifecycle.
* `ASPECT_TYPE_ID`: Custom global governance aspect type (`"pii"`).

> ℹ️ **Data Residency & Endpoint Decoupling**:
> In enterprise architectures, data storage regions (`DATA_LOCATION = "us"`) and AI model inference endpoints (`GEMINI_LOCATION = "global"`) are decoupled. While BigQuery stores customer data at rest within the designated US multi-region boundary, Gemini prompt evaluation occurs via the global inference gateway to access distributed TPU clusters. If your organization operates under sovereign data residency mandates or strict VPC Service Controls requiring processing within a single regional boundary, verify regional model availability in Google Cloud documentation and set `GEMINI_LOCATION` to your compliant region (such as `us-central1`).

In [ ]:
# @title Configuration & Parameter Initialization
PROJECT_ID = "your-project-id"  # @param {type:"string"}
DATA_LOCATION = "us"
KC_LOCATION = "us"
# Set to "global" for latest flagship models. If data residency compliance requires regional processing, set to your region (such as "us-central1"):
GEMINI_LOCATION = "global"
MODEL_NAME = "gemini-3.7-flash"
DATASET_ID = "thelook_ecommerce"
ASPECT_TYPE_ID = "pii"

if not PROJECT_ID or PROJECT_ID == "your-project-id" or PROJECT_ID.startswith("your-"):
    raise ValueError(
        "PROJECT_ID must be set to a valid Google Cloud project ID before executing downstream cells."
    )

print("Configuration parameters initialized:")
print(f"  PROJECT_ID:      {PROJECT_ID}")
print(f"  DATA_LOCATION:   {DATA_LOCATION}")
print(f"  KC_LOCATION:     {KC_LOCATION}")
print(f"  GEMINI_LOCATION: {GEMINI_LOCATION}")
print(f"  MODEL_NAME:      {MODEL_NAME}")
print(f"  DATASET_ID:      {DATASET_ID}")
print(f"  ASPECT_TYPE_ID:  {ASPECT_TYPE_ID}")


### Initialize Google Cloud SDK clients and declare structured data contracts

Instantiate native SDK clients for Knowledge Catalog, BigQuery, and the unified Google GenAI SDK, and declare structured Pydantic data schemas.

### Dual schema architecture: client-side models vs. catalog templates

Enterprise governance pipelines maintain dual schema representations for distinct operational purposes:
1. **Client-side Pydantic schemas**: `PiiColumnFinding` and `ComplianceAuditReport` define strongly typed contracts for client-side policy audits. `GroundedAgentDecision` defines the structured JSON response contract for Gemini, constraining AI outputs to verified table names, columns, and certified join conditions.
2. **Server-side AspectType templates**: Knowledge Catalog aspect types define the authoritative control-plane metadata schemas attached to catalog entries in Google Cloud.

> ℹ️ **Structured Output Enforcement**: Passing `GroundedAgentDecision` directly to `types.GenerateContentConfig(response_schema=...)` in the Google GenAI SDK instructs Gemini to output validated JSON conforming strictly to your Pydantic model, eliminating unstructured markdown parsing errors.

In [ ]:
# Initialize native Google Cloud SDK clients and structured schema models
from google import genai
from google.cloud import bigquery
from google.cloud import dataplex_v1
from pydantic import BaseModel, Field

# 1. Knowledge Catalog client
catalog_client = dataplex_v1.CatalogServiceClient()

# 2. BigQuery client
bq_client = bigquery.Client(project=PROJECT_ID, location=DATA_LOCATION)

# 3. Unified Google GenAI client
gemini_client = genai.Client(
    vertexai=True,
    project=PROJECT_ID,
    location=GEMINI_LOCATION,
)


# Structured data models for compliance audits and agent decisions
class PiiColumnFinding(BaseModel):
    column_name: str = Field(description="Name of the physical table column")
    pii_type: str = Field(description="Classified PII category (such as EMAIL, NAME, ADDRESS)")
    masked: bool = Field(description="Whether the column has been de-identified or masked")


class ComplianceAuditReport(BaseModel):
    table_resource: str = Field(description="Full resource name of the inspected table entry")
    total_pii_columns: int = Field(description="Total columns tagged with PII aspects")
    unmasked_violations: list[PiiColumnFinding] = Field(
        default_factory=list, description="Unmasked PII columns requiring compliance remediation"
    )


class GroundedAgentDecision(BaseModel):
    query_intent: str = Field(description="Interpreted analytical user intent")
    required_tables: list[str] = Field(description="Authoritative tables needed for query")
    required_columns: list[str] = Field(description="Specific columns needed from catalog context")
    join_conditions: list[str] = Field(
        description="Certified cross-table join predicates extracted from context"
    )
    generated_sql: str = Field(description="Synthesized BigQuery SQL query")


print("Native Google Cloud SDK clients and Pydantic schemas successfully initialized.")


---

## Section 3: Data contracts, schemas & asset provisioning

### Provision BigQuery dataset and copy sample e-commerce tables

Create the analytical dataset in the US multi-region with a 24-hour auto-expiration lifecycle and copy bounded sample tables (`users`, `orders`, `order_items`, `products`) from the public dataset.

### Physical dataset lifecycle and bounded data observability

To simulate an authentic enterprise environment with relational dependencies and sensitive attributes, this step provisions a dedicated BigQuery analytical dataset and populates four core tables from the public synthetic e-commerce repository (`bigquery-public-data.thelook_ecommerce`):
* `users`: Customer profiles containing sensitive Personally Identifiable Information (PII) such as email addresses, personal names, and physical street addresses.
* `orders`: Order transaction headers and status tracking.
* `order_items`: Line-item transactional pricing linked to products.
* `products`: Product catalog classifications and retail pricing.

Because `bigquery-public-data` is hosted in the BigQuery `US` multi-region, creating your destination dataset in `DATA_LOCATION = "us"` enables direct intra-region table copying via `bq_client.copy_table()` without cross-region network egress or Cloud Storage staging.

> ℹ️ **Resource Lifecycle Guardrail**: Setting `default_table_expiration_ms = 86400000` (24 hours) ensures that demonstration tables automatically expire even if an interactive session terminates before executing teardown.

Following table copy, an explicit SQL query prints the first 5 records from `{PROJECT_ID}.thelook_ecommerce.users` to confirm physical schema structure before attaching catalog governance aspects.

In [ ]:
# Provision BigQuery dataset and copy sample tables from public dataset
dataset_ref = f"{PROJECT_ID}.{DATASET_ID}"
dataset = bigquery.Dataset(dataset_ref)
dataset.location = DATA_LOCATION
dataset.default_table_expiration_ms = 86400000  # 24 hours auto-expiration
dataset = bq_client.create_dataset(dataset, exists_ok=True)
print(f"Provisioned BigQuery dataset: {dataset_ref} in location '{DATA_LOCATION}'.")

source_public_project = "bigquery-public-data"
source_dataset = "thelook_ecommerce"
sample_tables = ["users", "orders", "order_items", "products"]

copy_config = bigquery.CopyJobConfig(
    write_disposition=bigquery.WriteDisposition.WRITE_TRUNCATE
)

for table_name in sample_tables:
    source_table_ref = f"{source_public_project}.{source_dataset}.{table_name}"
    dest_table_ref = f"{dataset_ref}.{table_name}"
    copy_job = bq_client.copy_table(source_table_ref, dest_table_ref, job_config=copy_config)
    copy_job.result()
    print(f"Copied table '{source_table_ref}' -> '{dest_table_ref}'.")

# Preview sample records from users table to observe row structure
preview_query = f"""
SELECT id, first_name, last_name, email, age, gender, postal_code, street_address
FROM `{dataset_ref}.users`
LIMIT 5
"""
preview_df = bq_client.query(preview_query).to_dataframe()
print(f"\nPreviewing 5 records from `{dataset_ref}.users`:")
print(preview_df.to_string(index=False))

### Register custom global PII aspect type

Define and register a custom global AspectType (`pii`) in Knowledge Catalog with `pii_type` (ENUM) and `masked` (BOOL) fields using direct-return idempotency.

### AspectType blueprinting vs. concrete aspect instances

In Knowledge Catalog, metadata governance decouples the schema definition from attached metadata records:
* **AspectType**: The reusable governance blueprint defining allowed fields, types, and constraints in the Google Cloud control plane.
* **Aspect**: A concrete metadata document conforming to an AspectType, attached to a specific catalog entry or column.

Here, you register the `pii` aspect type under `projects/{PROJECT_ID}/locations/global`. Registering custom aspect types in the global control plane enables centralized discovery across all regional assets (such as BigQuery tables in `us` or Cloud Storage buckets in other regions) using a single, unified aspect schema identifier.

```
AspectType: projects/{PROJECT_ID}/locations/global/aspectTypes/pii
  ├── Field 1: pii_type (enum: EMAIL, NAME, ADDRESS, PHONE_NUMBER, DEMOGRAPHIC, OTHER) [required]
  └── Field 2: masked   (bool: true / false) [required]
```

### Procedural idempotency and direct-return mechanics (ADR-002)

To ensure this cell can be safely re-executed without runtime errors or duplicate creation attempts, `get_or_create_pii_aspect_type()` implements the direct-return idempotency pattern:
1. Proactively queries `catalog_client.get_aspect_type()` to inspect existing state.
2. If `NotFound` is raised, it issues `catalog_client.create_aspect_type()`.
3. If concurrent execution causes an `AlreadyExists` exception, it catches the exception and returns the live remote object directly from the catalog.

> 💡 **Tip: Integer Field Indices in Metadata Templates**:
> Knowledge Catalog metadata templates require explicit numeric indices (`index=1`, `index=2`) for every field and enum value. These indices preserve backwards-compatible Protobuf binary serialization when schemas evolve over time.

In [ ]:
# Register custom global PII AspectType with direct-return idempotency
import time
from google.api_core.exceptions import AlreadyExists, NotFound, ResourceExhausted

parent_location = f"projects/{PROJECT_ID}/locations/global"
aspect_type_name = f"{parent_location}/aspectTypes/{ASPECT_TYPE_ID}"


def get_or_create_pii_aspect_type():
    """Retrieve existing AspectType or create new global PII AspectType."""
    try:
        existing = catalog_client.get_aspect_type(name=aspect_type_name)
        print(f"AspectType '{aspect_type_name}' already exists. Reusing existing definition.")
        return existing
    except NotFound:
        print(f"AspectType '{aspect_type_name}' not found. Creating new definition...")

    aspect_type_payload = dataplex_v1.AspectType(
        description="Governance aspect tracking Personally Identifiable Information classification and masking status.",
        metadata_template=dataplex_v1.AspectType.MetadataTemplate(
            name="pii_metadata",
            type_="record",
            record_fields=[
                dataplex_v1.AspectType.MetadataTemplate(
                    name="pii_type",
                    type_="enum",
                    index=1,
                    enum_values=[
                        dataplex_v1.AspectType.MetadataTemplate.EnumValue(name="EMAIL", index=1),
                        dataplex_v1.AspectType.MetadataTemplate.EnumValue(name="NAME", index=2),
                        dataplex_v1.AspectType.MetadataTemplate.EnumValue(name="ADDRESS", index=3),
                        dataplex_v1.AspectType.MetadataTemplate.EnumValue(
                            name="PHONE_NUMBER", index=4
                        ),
                        dataplex_v1.AspectType.MetadataTemplate.EnumValue(
                            name="DEMOGRAPHIC", index=5
                        ),
                        dataplex_v1.AspectType.MetadataTemplate.EnumValue(name="OTHER", index=6),
                    ],
                    constraints=dataplex_v1.AspectType.MetadataTemplate.Constraints(required=True),
                ),
                dataplex_v1.AspectType.MetadataTemplate(
                    name="masked",
                    type_="bool",
                    index=2,
                    constraints=dataplex_v1.AspectType.MetadataTemplate.Constraints(required=True),
                ),
            ],
        ),
    )

    create_request = dataplex_v1.CreateAspectTypeRequest(
        parent=parent_location,
        aspect_type_id=ASPECT_TYPE_ID,
        aspect_type=aspect_type_payload,
    )

    try:
        operation = catalog_client.create_aspect_type(request=create_request)
        created = operation.result()
        print(f"Created custom AspectType: {created.name}")
        return created
    except AlreadyExists:
        print(f"AspectType '{aspect_type_name}' was created concurrently. Fetching remote state...")
        return catalog_client.get_aspect_type(name=aspect_type_name)


pii_aspect_type = get_or_create_pii_aspect_type()
print(f"Active AspectType name: {pii_aspect_type.name}")


### Attach column governance aspects and poll search index

Bind the custom `pii` aspect to specific columns of the `users` table, creating 4 unmasked violations and 3 masked compliant entries, and execute a bounded polling loop to guarantee search index propagation.

### BigQuery system entries and column-level aspect binding

Knowledge Catalog automatically harvests BigQuery datasets and tables into Google-managed system entries under the `@bigquery` entry group in your regional data plane (`locations/us`).

To audit column-level compliance, you bind the `pii` aspect to individual column nodes within the physical table schema using dot-separated and `@Schema` aspect key syntax:
`{PROJECT_ID}.global.pii@Schema.{column_name}`

In this step, seven columns are tagged with governance metadata:
* **4 unmasked compliance violations**: `email` (`EMAIL`), `first_name` (`NAME`), `last_name` (`NAME`), `street_address` (`ADDRESS`) with `masked = False`.
* **3 compliant de-identified columns**: `postal_code` (`ADDRESS`), `age` (`DEMOGRAPHIC`), `gender` (`DEMOGRAPHIC`) with `masked = True`.

> ⚠️ **Important: Scoped Update Masks on System Entries**:
> Knowledge Catalog BigQuery system entries are composite records that bundle Google-managed metadata (physical column schemas, table row counts, partition keys) with user-defined aspects. Calling `catalog_client.update_entry()` without scoping attempts to overwrite Google-managed fields, triggering a `403 PermissionDenied` error. Passing `update_mask=FieldMask(paths=['aspects'])` and `aspect_keys=list(user_aspects.keys())` guarantees that mutations are strictly isolated to your custom aspect keys.

### Asynchronous search index propagation and bounded polling (ADR-003)

In distributed cloud architectures, control-plane mutations persist immediately to Google Cloud Spanner transactional storage. However, global search indices ingest Change Data Capture (CDC) event streams asynchronously. This eventual consistency introduces a 30–90 second replication window before newly attached aspects become searchable.

To prevent downstream search cells from returning intermittent empty results, a bounded polling loop checks `catalog_client.search_entries()` up to 12 times at 10-second intervals until the table is confirmed discoverable in the search index.

> ℹ️ **Cloud Mechanics Behind the Scenes: Eventual Consistency (ADR-003)**:
> Automated enterprise pipelines must never rely on arbitrary hardcoded sleep statements (`time.sleep(60)`). Bounded polling loops with informative progress logging verify remote state deterministically while avoiding unnecessary pipeline delays.

In [ ]:
# Attach column aspects to BigQuery tables and execute bounded polling loop
import time
from google.api_core.exceptions import PermissionDenied
from google.protobuf import field_mask_pb2

# Knowledge Catalog BigQuery system entry names in regional data plane
bq_entry_name_users = f"projects/{PROJECT_ID}/locations/{KC_LOCATION}/entryGroups/@bigquery/entries/bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{DATASET_ID}/tables/users"
bq_entry_name_orders = f"projects/{PROJECT_ID}/locations/{KC_LOCATION}/entryGroups/@bigquery/entries/bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{DATASET_ID}/tables/orders"
bq_entry_name_order_items = f"projects/{PROJECT_ID}/locations/{KC_LOCATION}/entryGroups/@bigquery/entries/bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{DATASET_ID}/tables/order_items"
bq_entry_name_products = f"projects/{PROJECT_ID}/locations/{KC_LOCATION}/entryGroups/@bigquery/entries/bigquery.googleapis.com/projects/{PROJECT_ID}/datasets/{DATASET_ID}/tables/products"

# Wait for BigQuery system entry harvesting if table was freshly copied
print(f"Checking Knowledge Catalog BigQuery system entry for users table: `{bq_entry_name_users}`...")
for attempt in range(12):
    try:
        catalog_client.get_entry(name=bq_entry_name_users)
        print("BigQuery system entry is active in Knowledge Catalog.")
        break
    except NotFound:
        print(f"Waiting for system entry harvesting (attempt {attempt + 1}/12)...")
        time.sleep(5)

# Configure 7 column-level PII aspect instances (4 unmasked violations, 3 masked compliant)
user_columns_metadata = [
    ("email", "EMAIL", False),
    ("first_name", "NAME", False),
    ("last_name", "NAME", False),
    ("street_address", "ADDRESS", False),
    ("postal_code", "ADDRESS", True),
    ("age", "DEMOGRAPHIC", True),
    ("gender", "DEMOGRAPHIC", True),
]

user_aspects = {}
for col_name, p_type, is_masked in user_columns_metadata:
    aspect_key = f"{PROJECT_ID}.global.{ASPECT_TYPE_ID}@Schema.{col_name}"
    user_aspects[aspect_key] = dataplex_v1.Aspect(
        aspect_type=aspect_type_name,
        data={
            "pii_type": p_type,
            "masked": is_masked,
        },
    )

entry_users = dataplex_v1.Entry(
    name=bq_entry_name_users,
    aspects=user_aspects,
)

# Update entry using required UpdateEntryRequest wrapper and field mask
update_req = dataplex_v1.UpdateEntryRequest(
    entry=entry_users,
    update_mask=field_mask_pb2.FieldMask(paths=["aspects"]),
    aspect_keys=list(user_aspects.keys()),
)

# Attach column aspects with bounded retry loop for global aspect type cross-region replication
for update_attempt in range(12):
    try:
        catalog_client.update_entry(request=update_req)
        print(f"Successfully attached {len(user_aspects)} column-level PII aspects to `{bq_entry_name_users}`.")
        break
    except (PermissionDenied, NotFound) as update_err:
        if update_attempt < 11:
            print(f"Waiting for global aspect type cross-region replication (attempt {update_attempt + 1}/12)...")
            time.sleep(5)
        else:
            raise

# Bounded polling loop to verify search index propagation
print("\nInitiating bounded polling loop for search index propagation...")
search_poll_query = f"system=bigquery dataset={DATASET_ID} aspect:{PROJECT_ID}.global.{ASPECT_TYPE_ID}"
propagation_ready = False

for poll_attempt in range(12):
    search_poll_req = dataplex_v1.SearchEntriesRequest(
        name=f"projects/{PROJECT_ID}/locations/global",
        scope=f"projects/{PROJECT_ID}",
        query=search_poll_query,
        page_size=10,
    )
    search_poll_res = catalog_client.search_entries(request=search_poll_req)
    found_entries = [r.dataplex_entry.name for r in search_poll_res if r.dataplex_entry]
    if any("tables/users" in name for name in found_entries):
        propagation_ready = True
        print(f"✓ Search index propagation verified on attempt {poll_attempt + 1}. Discovered {len(found_entries)} matching entry.")
        break
    print(f"Waiting for global search index synchronization (attempt {poll_attempt + 1}/12)...")
    time.sleep(10)

if not propagation_ready:
    print("Notice: Search index propagation is still synchronizing asynchronously. Downstream lookup will inspect direct entry.")

---

## Section 4: 3-tier retrieval stack execution

### Tier 1: Semantic discovery and structured predicate queries

Execute natural-language semantic searches using the global control-plane endpoint (`semantic_search=True`) and contrast against unbounded predicate queries. Verify that search results intentionally return empty `aspects: {}` dictionaries.

### Dual query execution pathways in Knowledge Catalog search

Knowledge Catalog search operates on two distinct query execution engines:

| Feature Dimension | Semantic Natural-Language Search | Structured Predicate Query |
| :--- | :--- | :--- |
| **Query Format** | Free-text natural language (such as `"customer personal contact details"`) | Structured key-value predicates (`system=bigquery dataset=... aspect:...`) |
| **Backend Engine** | 2-stage neural model: Bi-Encoder vector ANN retrieval + Cross-Encoder re-ranker | Google Cloud Spanner deterministic index scan |
| **Candidate Cap** | **Hard capped at ~100 candidate entries** due to ANN index limits | **Unbounded**. Paginates through 10,000+ entries using `page_size` and `page_token` |
| **Primary Use Case** | Exploratory search, human discovery, AI agent query routing | Comprehensive compliance audits, catalog-wide asset enumeration |
| **Flag Requirement** | `semantic_search=True` | `semantic_search=True` (or omitted) |

To optimize performance, `name` targets the centralized global control plane (`locations/global`), while `scope` explicitly restricts candidate evaluation to the project boundary (`projects/{PROJECT_ID}`). Project-level scoping reduces query latency by up to 5x by avoiding global tenant sweeps.

### The empty aspects invariant (b/359193123)

A frequent point of disorientation for developers is observing that `result.dataplex_entry.aspects` returns an empty dictionary `{}` from `search_entries`.

> ℹ️ **Cloud Mechanics Behind the Scenes: The Empty Aspects Invariant (b/359193123)**:
> Search indices are optimized for high-throughput candidate filtering and ranking across millions of resources. Including heavy aspect payloads (such as column schemas, classifications, and lineage graphs) in search results would bloat search index memory, cause network serialization bottlenecks, and create side-channel security risks. Consequently, `SearchEntriesResponse` returns metadata references, intentionally omitting attached aspect payloads. To inspect column-level aspects, discovery must be chained to `lookup_entry` in Tier 2.

In [ ]:
# Tier 1: Execute semantic search and structured predicate queries via search_entries

# 1. Semantic Natural-Language Search (Neural vector ANN candidate generation)
print("--- 1. Executing Semantic Natural-Language Search ---")
semantic_query = "customer personal contact details and residential addresses"
semantic_req = dataplex_v1.SearchEntriesRequest(
    name=f"projects/{PROJECT_ID}/locations/global",
    scope=f"projects/{PROJECT_ID}",
    query=semantic_query,
    page_size=10,
    semantic_search=True,
)
semantic_res = list(catalog_client.search_entries(request=semantic_req))
discovered_semantic_entries = []

for result in semantic_res:
    entry_name = result.dataplex_entry.name if result.dataplex_entry else ""
    discovered_semantic_entries.append(entry_name)
    aspect_map = dict(result.dataplex_entry.aspects) if result.dataplex_entry else {}
    print(f"Discovered Entry: {entry_name}")
    print(f"  Linked Resource: {result.linked_resource}")
    print(f"  Aspects map count: {len(aspect_map)} (Empty: {aspect_map == {}})")

search_aspects_were_empty = all(
    (not r.dataplex_entry.aspects) for r in semantic_res if r.dataplex_entry
)
print(f"\nEmpirical Invariant Verified: search_entries returned empty aspects dictionary: {search_aspects_were_empty}")

# 2. Structured Predicate Scan (Unbounded Spanner index scan)
print("\n--- 2. Executing Structured Predicate Query ---")
predicate_query = f"system=bigquery dataset={DATASET_ID} aspect:{PROJECT_ID}.global.{ASPECT_TYPE_ID}"
predicate_req = dataplex_v1.SearchEntriesRequest(
    name=f"projects/{PROJECT_ID}/locations/global",
    scope=f"projects/{PROJECT_ID}",
    query=predicate_query,
    page_size=10,
)
predicate_res = list(catalog_client.search_entries(request=predicate_req))
discovered_predicate_entries = []

for result in predicate_res:
    entry_name = result.dataplex_entry.name if result.dataplex_entry else ""
    discovered_predicate_entries.append(entry_name)
    print(f"Structured Match: {entry_name}")

print(f"Structured predicate discovery identified {len(discovered_predicate_entries)} entries.")

### Tier 2: Deep inspection and column-level compliance audit

Chain discovered search resources into `lookup_entry` with `EntryView.CUSTOM` and `aspect_types` filtering. Perform client-side post-filtering to detect unmasked PII columns.

### Chaining discovery to deep inspection

To resolve the empty aspects invariant, you chain the discovered table resource name into `catalog_client.lookup_entry()`.

Targeting the regional data plane (`locations/us`), you configure two critical parameters on `LookupEntryRequest`:
1. `view = dataplex_v1.EntryView.CUSTOM`: Default lookup requests return basic entry attributes while omitting custom aspect payloads to conserve network bandwidth. Requesting `EntryView.CUSTOM` instructs Knowledge Catalog to serialize custom attached aspects.
2. `aspect_types = [aspect_type_name]`: Filters the response to your target governance schema, isolating PII aspects from extraneous system metadata.

### The aspect field search predicate degradation trap (b/529248015)

A common architectural trap is attempting to search directly for field-level values in `search_entries` (such as `aspect:...pii.masked=false` or `aspect:...pii.pii_type=EMAIL`).

> ⚠️ **Important: The Aspect Field Predicate Degradation Trap (b/529248015)**:
> In Knowledge Catalog search syntax, aspect predicates support catalog-level aspect type existence (`aspect:{PROJECT_ID}.global.pii`), but do not support filtering on nested aspect field values. When an unsupported field predicate like `aspect:...pii.masked=false` is passed, the search query parser silently strips the predicate syntax and treats the tokens `masked` and `false` as conversational free-text keywords. This unintentionally activates the two-stage neural search engine with its ~100 candidate cap, returning false-positive matches.
> 
> **Architectural Standard**: Use `search_entries` to discover candidate assets using aspect existence predicates, and perform precise field-level filtering (`is_masked is False`) client-side after executing `lookup_entry`.

Following inspection, the script builds a structured `ComplianceAuditReport` listing exactly the 4 unmasked PII columns (`email`, `first_name`, `last_name`, `street_address`) requiring masking remediation.

In [ ]:
# Tier 2: Deep inspection and column-level compliance audit via lookup_entry
users_entry_res = bq_entry_name_users

# Chain discovered search resource to deep lookup with custom entry view
lookup_req = dataplex_v1.LookupEntryRequest(
    name=f"projects/{PROJECT_ID}/locations/{KC_LOCATION}",
    entry=users_entry_res,
    view=dataplex_v1.EntryView.CUSTOM,
    aspect_types=[aspect_type_name],
)
live_users_entry = catalog_client.lookup_entry(request=lookup_req)
print(f"Successfully retrieved deep entry metadata for: {live_users_entry.name}")
print(f"Attached aspects count: {len(live_users_entry.aspects)}")

# Perform client-side post-filtering to detect unmasked PII columns
unmasked_findings = []
for k, v in live_users_entry.aspects.items():
    if ASPECT_TYPE_ID in k:
        col = k.split("@")[-1].replace("Schema.", "") if "@" in k else "table_level"
        aspect_data = dict(v.data)
        p_type = str(aspect_data.get("pii_type", "UNKNOWN"))
        is_masked = bool(aspect_data.get("masked", False))
        if not is_masked:
            unmasked_findings.append(
                PiiColumnFinding(column_name=col, pii_type=p_type, masked=False)
            )

audit_report = ComplianceAuditReport(
    table_resource=live_users_entry.name,
    total_pii_columns=len(live_users_entry.aspects),
    unmasked_violations=unmasked_findings,
)

print("\n=== Knowledge Catalog Compliance Audit Report ===")
print(f"Table Resource:       {audit_report.table_resource}")
print(f"Total Tagged Columns: {audit_report.total_pii_columns}")
print(f"Unmasked Violations:  {len(audit_report.unmasked_violations)}")
print("-" * 55)
print(f"{'Column Name':<20} | {'PII Classification':<18} | {'Masked Status'}")
print("-" * 55)
for finding in audit_report.unmasked_violations:
    print(f"{finding.column_name:<20} | {finding.pii_type:<18} | {finding.masked}")
print("-" * 55)

### Negative falsification gate: defensive empty candidate guard

Test the defensive input guard against empty candidate resource lists (`resources=[]`) and verify authentic Google Cloud error rejection (`CONTEXT_RESOURCES_EMPTY`).

### The CONTEXT_RESOURCES_EMPTY backend failure mode (b/529248015)

In autonomous agent architectures, an AI model dynamically generates search queries to locate relevant tables. When a search query produces zero candidate matches, an agent tool might pass an empty list (`resources=[]`) directly to `LookupContextRequest`.

> 🚨 **Warning: The CONTEXT_RESOURCES_EMPTY Backend Crash (b/529248015)**:
> In the Knowledge Catalog control plane, invoking `lookup_context` with an empty candidate list (`resources=[]`) results in an unhandled server-side validation exception:
> `google.api_core.exceptions.InvalidArgument: 400 resources must not be empty (CONTEXT_RESOURCES_EMPTY)`
> Without defensive client-side validation, this crash terminates the agent session and breaks automated workflow execution.

### Negative falsification proof and service boundary validation (ADR-016)

To protect agent pipelines, `get_context()` implements an explicit defensive input guard:
```python
if not candidate_resources:
    return "No matching catalog entries found."
```

This step verifies both sides of the contract:
1. **Defensive Guard Verification**: Passes an empty list `[]` to `get_context()` and asserts that the guard safely intercepts execution without calling the remote API.
2. **Authentic Service Boundary Rejection**: Bypasses the guard and calls `catalog_client.lookup_context()` with `resources=[]`, asserting that the live Google Cloud service raises `InvalidArgument` with `CONTEXT_RESOURCES_EMPTY`.

In [ ]:
# Negative Falsification Gate: test defensive guard and verify authentic cloud error rejection
import google.api_core.exceptions


def get_context(candidate_resources: list[str]) -> str:
    """Safely retrieves Knowledge Catalog context with defensive guard against empty candidate lists."""
    if not candidate_resources:
        print("Defensive Guard Activated: candidate_resources list is empty. Early exit triggered.")
        return "No matching catalog entries found."

    request = dataplex_v1.LookupContextRequest(
        name=f"projects/{PROJECT_ID}/locations/{KC_LOCATION}",
        resources=candidate_resources[:10],
        options={"format": "yaml", "context_budget": "8000"},
    )
    response = catalog_client.lookup_context(request=request)
    return response.context


# 1. Test defensive guard with empty candidate resources
print("--- 1. Testing Defensive Input Guard ---")
guard_output = get_context([])
print(f"Guard return value: '{guard_output}'")
assert guard_output == "No matching catalog entries found.", "Defensive guard failed to intercept empty candidate list"

# 2. Negative test: verify authentic cloud service rejection when guard is bypassed
print("\n--- 2. Testing Authentic Cloud Service Boundary Rejection ---")
try:
    bad_request = dataplex_v1.LookupContextRequest(
        name=f"projects/{PROJECT_ID}/locations/{KC_LOCATION}",
        resources=[],
    )
    catalog_client.lookup_context(request=bad_request)
    print("Warning: Remote service unexpectedly accepted empty resources.")
except google.api_core.exceptions.InvalidArgument as exc:
    print(f"✓ Verified authentic cloud rejection: {exc.message}")
    assert "resources must not be empty" in str(exc) or "CONTEXT_RESOURCES_EMPTY" in str(exc) or "InvalidArgument" in str(type(exc).__name__)


### Tier 3: Operational context hydration and schema extraction

Hydrate prompt-ready operational and technical schema context using `lookup_context` formatted as YAML bounded by a character budget.

### Prompt-ready context hydration vs. raw schema dumps

When providing catalog metadata to Large Language Models (LLMs), passing raw JSON database schemas or DDL dumps consumes excessive prompt tokens and forces the model to guess join relationships.

Knowledge Catalog `lookup_context` solves this by synthesizing two layers of metadata into a prompt-optimized document:
1. **Technical schema definitions**: Column names, data types, column modes, and semantic field descriptions.
2. **Operational query telemetry**: Frequent join paths (`frequent_joins`) discovered from enterprise query history, identifying verified foreign key join keys between tables (such as `order_items.product_id = products.id`).

```
LookupContextRequest
  ├── resources: [orders, order_items, products] (up to 10 entries)
  └── options:
        ├── format: "yaml"          (30-40% lower token overhead than JSON)
        └── context_budget: "8000"  (bounds prompt character length)
```

> 💡 **Tip: Token-Efficient YAML Formatting for LLM Context**:
> Selecting `"format": "yaml"` eliminates JSON syntax overhead (braces, commas, quotation marks) and maintains indentation hierarchy, helping transformer attention layers accurately parse nested schema structures while reducing LLM token consumption.

In [ ]:
# Tier 3: Operational context hydration and schema extraction via lookup_context
candidate_resources = [
    bq_entry_name_orders,
    bq_entry_name_order_items,
    bq_entry_name_products,
]

print(f"Hydrating Knowledge Catalog context for {len(candidate_resources)} analytical tables:")
for res in candidate_resources:
    print(f"  • {res}")

hydrated_yaml_context = get_context(candidate_resources)

print("\n=== Hydrated Operational Context (YAML) ===")
yaml_lines = hydrated_yaml_context.splitlines()
print("\n".join(yaml_lines[:40]))
if len(yaml_lines) > 40:
    print(f"... [{len(yaml_lines) - 40} additional context lines bounded by budget]")


### Autonomous AI agent grounding with Gemini 3.7 Flash

Ground Gemini 3.7 Flash on the hydrated Knowledge Catalog context to resolve natural-language analytics questions into certified cross-table SQL joins with zero hallucinations.

### Grounding generative AI on authoritative catalog metadata

Autonomous analytics agents frequently suffer from schema hallucinations when generating relational SQL queries from natural-language prompts. Without authoritative schema context, language models frequently:
* Invent non-existent column names (such as guessing `item_price` or `amount` instead of physical column `sale_price`).
* Hallucinate invalid cross-table join keys (such as joining on `orders.id = products.id` rather than traversing line items).
* Query stale, deprecated, or unauthorized datasets.

To eliminate hallucinations, the agent prompt injects the hydrated Knowledge Catalog YAML context as the authoritative boundary of truth.

```
User Query: "Compute total sales revenue by product category and provide certified join path"
                                  │
                                  ▼
                     Prompt + Catalog YAML Context
                                  │
                                  ▼
      Gemini 3.7 Flash (Gemini Enterprise Agent Platform)
                                  │
                                  ▼
                        GroundedAgentDecision
       ├── Query Intent: Revenue aggregation by product category
       ├── Required Tables: [order_items, products]
       ├── Required Columns: [sale_price, category, product_id, id]
       ├── Join Conditions: [order_items.product_id = products.id]
       └── Generated SQL: SELECT p.category, SUM(oi.sale_price)...
```

> ℹ️ **Eliminating SQL Hallucinations Through Catalog Telemetry**:
> Supplying Knowledge Catalog `frequent_joins` telemetry directly in the prompt grounds Gemini on certified enterprise join paths (`order_items.product_id = products.id`), ensuring synthesized BigQuery SQL executes cleanly without manual developer debugging.

In [ ]:
# Autonomous AI Agent Grounding with Gemini 3.7 Flash
from google.genai import types

prompt_text = f"""
You are an expert Google Cloud data analytics agent.
Use the following authoritative Knowledge Catalog metadata context to analyze the user query.
Extract the required tables, columns, and certified cross-table join predicates.
Synthesize a valid, optimized BigQuery SQL query using dataset `{PROJECT_ID}.{DATASET_ID}`.

AUTHORITATIVE CATALOG CONTEXT:
{hydrated_yaml_context}

USER ANALYTICAL QUERY:
Which tables and columns do I need to query to compute total sales revenue by product category, and what is the exact join path? Provide the certified BigQuery SQL query.
"""

agent_response = gemini_client.models.generate_content(
    model=MODEL_NAME,
    contents=prompt_text,
    config=types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=GroundedAgentDecision,
    ),
)

import json
agent_decision = GroundedAgentDecision.model_validate_json(agent_response.text)

print("=== Grounded AI Agent Reasoning & Decision ===")
print(f"Query Intent:       {agent_decision.query_intent}")
print(f"Required Tables:    {agent_decision.required_tables}")
print(f"Required Columns:   {agent_decision.required_columns}")
print(f"Join Conditions:    {agent_decision.join_conditions}")
print("\nSynthesized BigQuery SQL:")
print(agent_decision.generated_sql)


---

## Section 5: Multi-level integrity assertions & resilient teardown

### Verify governance audit and agent query correctness

Execute substantive integrity assertions confirming compliance audit precision, aspect schema invariants, and grounded SQL join path accuracy.

### Substantive verification gates

To validate that your metadata governance and AI grounding pipelines operate with mathematical precision, this step evaluates four automated integrity assertions:
1. **Compliance Audit Precision**: Asserts that `audit_report.unmasked_violations` identifies exactly the four intended unmasked PII columns (`email`, `first_name`, `last_name`, `street_address`).
2. **Aspect Attachment Completeness**: Confirms `audit_report.total_pii_columns == 7`, verifying that both masked and unmasked column aspects remain active in Knowledge Catalog.
3. **AI Grounding Fidelity**: Verifies that the synthesized SQL query references certified physical tables (`order_items`, `products`), metric columns (`sale_price`), dimension columns (`category`), and join keys (`product_id`).
4. **Search Invariant Verification**: Formally confirms that `search_aspects_were_empty` evaluated to `True`, verifying that the catalog decoupled lightweight discovery from deep inspection.

In [ ]:
# Multi-level substantive integrity assertions
print("Executing multi-level substantive integrity assertions...")

# 1. Assert compliance audit identified exactly 4 unmasked PII violations
assert len(audit_report.unmasked_violations) == 4, (
    f"Integrity check failed: expected 4 unmasked PII violations, but found {len(audit_report.unmasked_violations)}."
)
violation_columns = {finding.column_name for finding in audit_report.unmasked_violations}
expected_violations = {"email", "first_name", "last_name", "street_address"}
assert violation_columns == expected_violations, (
    f"Integrity check failed: violation columns {violation_columns} do not match expected {expected_violations}."
)
print("✓ Compliance audit assertions passed: 4 unmasked PII columns accurately detected.")

# 2. Assert total tagged columns equals 7
assert audit_report.total_pii_columns == 7, (
    f"Integrity check failed: expected 7 total tagged columns, found {audit_report.total_pii_columns}."
)
print("✓ Aspect attachment assertions passed: 7 total governance aspects confirmed.")

# 3. Assert agent join-key accuracy and table resolution
sql_lower = agent_decision.generated_sql.lower()
assert "order_items" in sql_lower, "Grounded agent query missing 'order_items' table."
assert "products" in sql_lower, "Grounded agent query missing 'products' table."
assert "sale_price" in sql_lower, "Grounded agent query missing 'sale_price' metric column."
assert "category" in sql_lower, "Grounded agent query missing 'category' dimension column."
assert "product_id" in sql_lower or "id" in sql_lower, "Grounded agent query missing join condition on product identifier."
print("✓ AI agent grounding assertions passed: cross-table join path resolved with zero hallucinations.")

# 4. Assert search empty aspects invariant
assert search_aspects_were_empty is True, "Search entries unexpectedly returned populated aspect maps."
print("✓ Empty aspects search invariant verified: discovery properly decoupled from deep inspection.")

print("\nAll multi-level integrity checks passed successfully.")


### Resilient standalone teardown and environment reset

Clean up all demonstration resources in reverse dependency order. This cell is fully self-contained and guarded to execute safely across initial, partial, or complete runs.

### Standalone idempotent teardown mechanics

To ensure safe execution across partial, interrupted, or repeated test runs, the teardown cell implements three core resilience patterns:
1. **Reverse Dependency Deletion**: Deletes downstream physical assets (BigQuery analytical dataset and tables) before removing upstream governance templates (Knowledge Catalog `AspectType`). Note that deleting the AspectType automatically cascades, detaching all bound column aspect instances across the catalog.
2. **Self-Contained Imports**: Explicitly imports required exception classes (`NotFound`, `ResourceExhausted`) within the teardown cell, preventing `NameError` crashes if executed in isolation.
3. **Universal State Guarding (`in locals()`)**: Validates that client and identifier variables exist in local memory before issuing delete RPCs, safely handling partially executed sessions.

> 🚨 **Warning: Irreversible Resource Deletion**:
> Executing the teardown cell permanently drops the BigQuery analytical dataset `{DATASET_ID}` and deletes the custom AspectType `{ASPECT_TYPE_ID}` from Knowledge Catalog. Only run this cell after completing all compliance audits and verification assertions.

In [ ]:
# Clean up demonstration resources in reverse dependency order
import time
from google.api_core.exceptions import NotFound, ResourceExhausted

print("Initiating resilient resource teardown...")

# 1. Delete BigQuery demonstration dataset and all tables
if "bq_client" in locals() and "dataset_ref" in locals() and dataset_ref:
    try:
        bq_client.delete_dataset(dataset_ref, delete_contents=True, not_found_ok=True)
        print(f"Deleted BigQuery dataset: {dataset_ref}")
    except Exception as e:
        print(f"Note on dataset deletion: {e}")

# 2. Delete Knowledge Catalog custom AspectType
if "catalog_client" in locals() and "aspect_type_name" in locals() and aspect_type_name:
    for attempt in range(5):
        try:
            op = catalog_client.delete_aspect_type(name=aspect_type_name)
            if hasattr(op, "result"):
                op.result()
            print(f"Deleted Knowledge Catalog AspectType: {aspect_type_name}")
            break
        except NotFound:
            print(f"AspectType already deleted or not found: {aspect_type_name}")
            break
        except ResourceExhausted:
            time.sleep(10)
        except Exception as e:
            print(f"Note on AspectType deletion: {e}")
            break

print("Teardown complete. Google Cloud environment cleanly reset.")


---

## Section 6: Summary and production bridging

### Key technical takeaways

1. **3-tier programmatic retrieval stack**:
   * **`search_entries` (Discovery tier)**: Evaluates broad natural-language and predicate queries over the global control plane (`locations/global`). Returns empty `aspects: {}` dictionaries by design.
   * **`lookup_entry` (Deep inspection tier)**: Targets regional data planes (`locations/us`) with `EntryView.CUSTOM` and `aspect_types` filters to inspect column-level metadata and audit policy compliance.
   * **`lookup_context` (Prompt hydration tier)**: Compresses technical schemas and frequent join telemetry into token-efficient YAML context bounded by character budgets, requiring defensive guards against empty candidate lists.
2. **Eventual consistency resilience**: Programmatic bounded polling loops intercept Change Data Capture (CDC) replication lag between physical asset creation and search index visibility.
3. **Hallucination elimination**: Grounding generative AI models on authoritative catalog schemas ensures SQL synthesis adheres strictly to enterprise physical contracts.

### Production bridging and enterprise deployment

To bridge this cookbook into enterprise production workflows:
* **Continuous compliance scanning**: Replace manual lookup scripts with automated profiling rules using Knowledge Catalog DataScan.
* **Real-time policy enforcement**: Stream BigQuery and catalog audit events via Cloud Audit Logs and Eventarc to trigger automated masking remediation workflows.
* **Enterprise data mesh governance**: Organize domain schemas across distributed data products governed by centralized policy tags.

### Documentation and next steps

* [Knowledge Catalog search documentation](https://cloud.google.com/dataplex/docs/search?utm_source=devrel&utm_medium=notebook&utm_campaign=catalog_search_and_retrieval)
* [Knowledge Catalog custom aspect types](https://cloud.google.com/dataplex/docs/use-aspect-types?utm_source=devrel&utm_medium=notebook&utm_campaign=catalog_search_and_retrieval)
* [Gemini Enterprise Agent Platform model reference](https://cloud.google.com/vertex-ai/generative-ai/docs/learn/models?utm_source=devrel&utm_medium=notebook&utm_campaign=catalog_search_and_retrieval)
* [Google GenAI SDK repository](https://github.com/googleapis/python-genai)